In [1]:
# =============================================================================
# NOTEBOOK 2: RESNET50 TRAINING + INFERENCE
# =============================================================================
# Uses DataParallel (works reliably in Kaggle notebooks)
# Attach Notebook 1's output as a Kaggle dataset input.
# GPU: T4 x2
# =============================================================================

import os
import gc
import random
import time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler
from torchvision.models import resnet50, ResNet50_Weights
from sklearn.metrics import f1_score
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

def seed_all(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = True

seed_all(42)

# =============================================================================
# HARDWARE
# =============================================================================
n_gpu = torch.cuda.device_count()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
for i in range(n_gpu):
    props = torch.cuda.get_device_properties(i)
    print(f"GPU {i}: {props.name} ({props.total_memory/1e9:.1f}GB)")

# =============================================================================
# PATHS - Adjust SPEC_DIR to match your Notebook 1 output dataset name
# =============================================================================
SPEC_DIR = Path('/kaggle/input/datasets/neerajsurin/data-10000/spectrograms')

assert (SPEC_DIR / 'train' / 'data.pt').exists(), \
    f"Train data not found! Searched: {SPEC_DIR}\nList /kaggle/input contents and adjust SPEC_DIR"

print(f"\nSpec dir: {SPEC_DIR}")

COMP_DIR = Path('/kaggle/input/competitions/jan-2026-dl-gen-ai-project/messy_mashup')
TEST_CSV = COMP_DIR / 'test.csv'
OUTPUT_DIR = Path('/kaggle/working')
OUTPUT_DIR.mkdir(exist_ok=True)

# Load config from Notebook 1
config_data = torch.load(SPEC_DIR / 'config.pt', weights_only=False)
GENRES = config_data['genres']
G2I = config_data['g2i']
I2G = {v: k for k, v in G2I.items()}
N_CLS = config_data['n_cls']
print(f"Config: {N_CLS} classes, genres={GENRES}")

# =============================================================================
# TRAINING CONFIG
# =============================================================================
class TrainCFG:
    epochs = 40
    bs = 128 * n_gpu if n_gpu >= 2 else 128  # 256 effective with 2 GPUs
    lr = 1e-3
    wd = 1e-4
    warmup = 0.05
    grad_clip = 1.0
    label_smooth = 0.05
    mixup_p = 0.3
    mixup_alpha = 0.3
    spec_aug_p = 0.5
    freq_mask = 20
    time_mask = 40
    n_freq_masks = 2
    n_time_masks = 2

tcfg = TrainCFG()
print(f"\nTraining: bs={tcfg.bs}, epochs={tcfg.epochs}, lr={tcfg.lr}")
print(f"Multi-GPU: {n_gpu >= 2} ({n_gpu} GPUs)")

# =============================================================================
# DATASET
# =============================================================================
class MelDataset(Dataset):
    def __init__(self, data_path, is_train=True):
        loaded = torch.load(data_path, weights_only=False)
        self.data = loaded['data']       # (N, 1, 224, 224)
        self.labels = loaded['labels']   # (N,)
        self.is_train = is_train
        print(f"  Loaded: {self.data.shape}, labels={self.labels.shape}")

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        mel = self.data[idx].clone()  # (1, 224, 224)

        # Repeat to 3 channels for pretrained ResNet50
        mel = mel.repeat(3, 1, 1)  # (3, 224, 224)

        # SpecAugment
        if self.is_train and random.random() < tcfg.spec_aug_p:
            mel = self._spec_augment(mel)

        label = self.labels[idx].item()

        # Soft label with smoothing
        lbl = torch.full((N_CLS,), tcfg.label_smooth / N_CLS)
        lbl[label] = 1.0 - tcfg.label_smooth + tcfg.label_smooth / N_CLS

        return mel, lbl, label

    def _spec_augment(self, spec):
        _, H, W = spec.shape
        spec = spec.clone()
        for _ in range(tcfg.n_freq_masks):
            f = random.randint(1, min(tcfg.freq_mask, H - 1))
            f0 = random.randint(0, H - f)
            spec[:, f0:f0+f, :] = 0
        for _ in range(tcfg.n_time_masks):
            t = random.randint(1, min(tcfg.time_mask, W - 1))
            t0 = random.randint(0, W - t)
            spec[:, :, t0:t0+t] = 0
        return spec

# =============================================================================
# MODEL
# =============================================================================
def create_model():
    model = resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)
    model.fc = nn.Sequential(
        nn.Dropout(0.3),
        nn.Linear(2048, N_CLS)
    )
    # Initialize FC
    for m in model.fc.modules():
        if isinstance(m, nn.Linear):
            nn.init.trunc_normal_(m.weight, std=0.02)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
    return model

# =============================================================================
# LOSS
# =============================================================================
class SoftCE(nn.Module):
    def forward(self, logits, targets):
        return -(targets * F.log_softmax(logits, dim=-1)).sum(dim=-1).mean()

# =============================================================================
# TRAINING
# =============================================================================
def train():
    print("\n" + "="*60)
    print("LOADING DATA")
    print("="*60)

    train_ds = MelDataset(SPEC_DIR / 'train' / 'data.pt', is_train=True)
    val_ds = MelDataset(SPEC_DIR / 'val' / 'data.pt', is_train=False)

    train_dl = DataLoader(train_ds, batch_size=tcfg.bs, shuffle=True,
                          num_workers=4, pin_memory=True, drop_last=True,
                          persistent_workers=True, prefetch_factor=3)
    val_dl = DataLoader(val_ds, batch_size=tcfg.bs, shuffle=False,
                        num_workers=4, pin_memory=True)

    print(f"Train: {len(train_ds)}, Val: {len(val_ds)}")
    print(f"Batches: train={len(train_dl)}, val={len(val_dl)}")

    print("\n" + "="*60)
    print("BUILDING MODEL")
    print("="*60)

    model = create_model()
    if n_gpu >= 2:
        model = nn.DataParallel(model)
        print(f"Using DataParallel with {n_gpu} GPUs")
    model = model.to(device)

    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Parameters: {n_params:,}")

    optimizer = torch.optim.AdamW(model.parameters(), lr=tcfg.lr, weight_decay=tcfg.wd,
                                   betas=(0.9, 0.999))

    total_steps = len(train_dl) * tcfg.epochs
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=tcfg.lr, total_steps=total_steps,
        pct_start=tcfg.warmup, anneal_strategy='cos',
        div_factor=25, final_div_factor=1000
    )

    criterion = SoftCE()
    scaler = GradScaler()

    best_f1 = 0.0
    best_epoch = 0
    patience = 0
    max_patience = 12

    print("\n" + "="*60)
    print("TRAINING")
    print("="*60)

    for epoch in range(tcfg.epochs):
        # ---- Train ----
        model.train()
        total_loss = 0
        n_batch = 0
        t0 = time.time()

        for mel, lbl_soft, lbl_hard in train_dl:
            mel = mel.to(device, non_blocking=True)
            lbl_soft = lbl_soft.to(device, non_blocking=True)

            # Mixup on batch
            if random.random() < tcfg.mixup_p:
                lam = np.random.beta(tcfg.mixup_alpha, tcfg.mixup_alpha)
                idx = torch.randperm(mel.size(0), device=mel.device)
                mel = lam * mel + (1 - lam) * mel[idx]
                lbl_soft = lam * lbl_soft + (1 - lam) * lbl_soft[idx]

            optimizer.zero_grad(set_to_none=True)

            with autocast():
                logits = model(mel)
                loss = criterion(logits, lbl_soft)

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), tcfg.grad_clip)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()

            total_loss += loss.item()
            n_batch += 1

        train_loss = total_loss / n_batch
        train_time = time.time() - t0

        # ---- Validate ----
        model.eval()
        all_preds = []
        all_tgts = []

        with torch.no_grad():
            for mel, lbl_soft, lbl_hard in val_dl:
                mel = mel.to(device, non_blocking=True)
                with autocast():
                    logits = model(mel)
                all_preds.extend(logits.argmax(dim=1).cpu().numpy())
                all_tgts.extend(lbl_hard.numpy())

        val_f1 = f1_score(all_tgts, all_preds, average='macro')
        per_cls = f1_score(all_tgts, all_preds, average=None)

        cls_str = " ".join(f"{GENRES[i][:3]}:{per_cls[i]:.3f}" for i in range(N_CLS))
        lr_now = scheduler.get_last_lr()[0]

        print(f"Ep {epoch+1:2d}/{tcfg.epochs} | loss={train_loss:.4f} | "
              f"F1={val_f1:.4f} | lr={lr_now:.2e} | {train_time:.0f}s")
        print(f"  {cls_str}")

        if val_f1 > best_f1:
            best_f1 = val_f1
            best_epoch = epoch + 1
            patience = 0
            state = model.module.state_dict() if hasattr(model, 'module') else model.state_dict()
            torch.save({
                'state': state,
                'f1': best_f1,
                'epoch': best_epoch
            }, OUTPUT_DIR / 'best_resnet50.pth')
            print(f"  -> New best: {val_f1:.4f}")
        else:
            patience += 1
            if patience >= max_patience:
                print(f"  Early stopping at epoch {epoch+1}")
                break

    print(f"\nTraining complete! Best F1: {best_f1:.4f} at epoch {best_epoch}")
    return best_f1

# Run training
best_f1 = train()

# Clean up training memory
gc.collect()
torch.cuda.empty_cache()

# =============================================================================
# INFERENCE WITH TTA
# =============================================================================
print("\n" + "="*60)
print("INFERENCE")
print("="*60)

# Load best model
model = resnet50(weights=None)
model.fc = nn.Sequential(
    nn.Dropout(0.3),
    nn.Linear(2048, N_CLS)
)

ckpt = torch.load(OUTPUT_DIR / 'best_resnet50.pth', map_location=device, weights_only=False)
model.load_state_dict(ckpt['state'])
model = model.to(device)
model.eval()
print(f"Loaded model: F1={ckpt['f1']:.4f}, epoch={ckpt['epoch']}")

# Load test spectrograms
test_data = torch.load(SPEC_DIR / 'test' / 'data.pt', weights_only=False)
test_mels = test_data['data']    # (N, 1, 224, 224)
test_ids = test_data['ids']      # (N,)
print(f"Test samples: {len(test_ids)}")

# TTA inference
def predict_tta(model, mels, batch_size=128):
    model.eval()
    all_probs = []

    n_tta = 5  # original + flip + 3 freq shifts

    with torch.no_grad():
        for i in range(0, len(mels), batch_size):
            batch = mels[i:i+batch_size]
            batch_3ch = batch.repeat(1, 3, 1, 1).to(device)  # (B, 3, 224, 224)

            probs_accum = torch.zeros(batch_3ch.size(0), N_CLS, device=device)

            # 1) Original
            with autocast():
                logits = model(batch_3ch)
            probs_accum += F.softmax(logits.float(), dim=1)

            # 2) Time reversal (horizontal flip)
            with autocast():
                logits = model(batch_3ch.flip(dims=[3]))
            probs_accum += F.softmax(logits.float(), dim=1)

            # 3-5) Small frequency shifts
            for shift in [-5, 3, 7]:
                shifted = torch.roll(batch_3ch, shifts=shift, dims=2)
                if shift > 0:
                    shifted[:, :, :shift, :] = 0
                else:
                    shifted[:, :, shift:, :] = 0
                with autocast():
                    logits = model(shifted)
                probs_accum += F.softmax(logits.float(), dim=1)

            avg_probs = probs_accum / n_tta
            all_probs.append(avg_probs.cpu())

    return torch.cat(all_probs, dim=0)

print("Running inference with TTA (5 augmentations)...")
probs = predict_tta(model, test_mels, batch_size=128)
predictions = probs.argmax(dim=1).numpy()

# Build submission
sub_ids = test_ids.numpy()
submission = pd.DataFrame({
    'id': sub_ids,
    'genre': [I2G[p] for p in predictions]
})
submission['id'] = submission['id'].astype(str).str.zfill(4)
submission = submission.sort_values('id')

sub_path = OUTPUT_DIR / 'submission.csv'
submission.to_csv(sub_path, index=False)

print(f"\nSaved: {sub_path}")
print(f"Predictions: {len(submission)}")

# Save probabilities
prob_df = pd.DataFrame(probs.numpy(), columns=GENRES)
prob_df.insert(0, 'id', [str(x).zfill(4) for x in sub_ids])
prob_df = prob_df.sort_values('id')
prob_df.to_csv(OUTPUT_DIR / 'probs.csv', index=False)

# Stats
print(f"\nDistribution:")
print(submission['genre'].value_counts().sort_index())

max_probs = probs.max(dim=1).values.numpy()
print(f"\nConfidence: mean={max_probs.mean():.4f}, min={max_probs.min():.4f}")
print(f"  >0.95: {(max_probs > 0.95).sum()}")
print(f"  >0.90: {(max_probs > 0.90).sum()}")
print(f"  <0.50: {(max_probs < 0.5).sum()}")

counts = submission['genre'].value_counts()
print(f"\nBalance: {counts.min()} - {counts.max()} (std={counts.std():.1f})")

print(f"\n{'='*60}")
print("COMPLETE!")
print(f"{'='*60}")

GPU 0: Tesla T4 (15.6GB)
GPU 1: Tesla T4 (15.6GB)

Spec dir: /kaggle/input/datasets/neerajsurin/data-10000/spectrograms
Config: 10 classes, genres=['blues', 'classical', 'country', 'disco', 'hiphop', 'jazz', 'metal', 'pop', 'reggae', 'rock']

Training: bs=256, epochs=40, lr=0.001
Multi-GPU: True (2 GPUs)

LOADING DATA
  Loaded: torch.Size([10000, 1, 224, 224]), labels=torch.Size([10000])
  Loaded: torch.Size([2000, 1, 224, 224]), labels=torch.Size([2000])
Train: 10000, Val: 2000
Batches: train=39, val=8

BUILDING MODEL
Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 203MB/s]


Using DataParallel with 2 GPUs
Parameters: 23,528,522

TRAINING
Ep  1/40 | loss=1.9969 | F1=0.4271 | lr=5.30e-04 | 98s
  blu:0.489 cla:0.298 cou:0.430 dis:0.196 hip:0.313 jaz:0.485 met:0.710 pop:0.627 reg:0.546 roc:0.179
  -> New best: 0.4271
Ep  2/40 | loss=0.9591 | F1=0.4287 | lr=1.00e-03 | 91s
  blu:0.297 cla:0.926 cou:0.067 dis:0.222 hip:0.661 jaz:0.713 met:0.625 pop:0.316 reg:0.378 roc:0.083
  -> New best: 0.4287
Ep  3/40 | loss=0.7772 | F1=0.7714 | lr=9.98e-04 | 91s
  blu:0.799 cla:0.978 cou:0.733 dis:0.737 hip:0.491 jaz:0.895 met:0.967 pop:0.804 reg:0.769 roc:0.541
  -> New best: 0.7714
Ep  4/40 | loss=0.6272 | F1=0.6409 | lr=9.93e-04 | 91s
  blu:0.288 cla:0.920 cou:0.610 dis:0.642 hip:0.833 jaz:0.671 met:0.786 pop:0.753 reg:0.438 roc:0.469
Ep  5/40 | loss=0.5370 | F1=0.7921 | lr=9.84e-04 | 91s
  blu:0.615 cla:0.998 cou:0.572 dis:0.826 hip:0.962 jaz:0.947 met:0.961 pop:0.954 reg:0.871 roc:0.214
  -> New best: 0.7921
Ep  6/40 | loss=0.4593 | F1=0.8152 | lr=9.73e-04 | 91s
  blu:0.